# Format GEO Datasets → CuMiDa CSV (series-matrix builder)

Builds analysis-ready datasets from raw NCBI GEO **series matrix** files
(`GSE*_series_matrix.txt.gz`) using `src/dataset_builder`.

For each dataset it writes **5 files** into the dataset's raw folder:

| File | Contents |
|---|---|
| `<Tissue>_<GEO>_probe.csv` | **Dataset 2** — probe-level CuMiDa (`samples, type, probe_1, …`) — feeds the benchmark |
| `<Tissue>_<GEO>_gene.csv`  | **Dataset 1** — gene-level CuMiDa (probes collapsed to gene symbols) |
| `<GEO>_metadata.csv`       | all per-sample characteristics |
| `<GEO>_mapping.csv`        | `probID \| gene_symbol \| source \| dedup_status` — every original probe ID, including AFFX-* controls (marked `dedup_status="Removed_AFFX"`) |
| `<GEO>_annotation_report.txt` | coverage stats (annotation rows / mapped / unmapped / AFFX removed / unique probes & genes) |

Probe → gene mapping: local GPL annotation file first (offline), MyGene.info fallback.
Affymetrix control probes (`AFFX-*`) that never resolve to a gene are excluded
from the probe-level CSV, but stay listed in `<GEO>_mapping.csv` with
`dedup_status="Removed_AFFX"` — no probe silently disappears without a trace.

**Config-driven:** build parameters for each raw GEO series live in
`configs/dataset_origin.yaml` — edit that file to add a new dataset, then run
this notebook. (Run-time dataset registration — `enabled`, `platform`,
`class_labels`, … — lives separately in `configs/datasets.yaml`.)

## 1. Paths & imports

In [ ]:
import sys
from pathlib import Path

# Run this notebook from the notebooks/ directory.
REPO_ROOT = Path("..").resolve()                  # repo root (parent of notebooks/)
sys.path.insert(0, str(REPO_ROOT))                # make src.* importable

import pandas as pd
from src.helper import ConfigLoader, load_dataset
from src.dataset_builder import (
    parse_series_matrix,
    discover_class_characteristics,
    build_geo_dataset,
)

cfg = ConfigLoader(config_root=str(REPO_ROOT / "configs"))
print("Repo root:", REPO_ROOT)

## 2. Collect buildable datasets from config

Every entry in `configs/dataset_origin.yaml` is a buildable raw GEO series.
Each `build_geo_dataset()` call produces both the probe-level and gene-level
CuMiDa CSVs at once.

In [ ]:
origins_cfg = cfg.load_dataset_origin_config()["origins"]

build_specs = {}
for gse_id, oc in origins_cfg.items():
    origin = cfg.get_origin_config(gse_id)
    build_specs[gse_id] = {
        "gse_id": gse_id,
        "series_matrix": Path(origin["series_matrix"]),
        "annotation_file": Path(origin["annotation_file"]) if origin.get("annotation_file") else None,
        "class_characteristic": origin.get("class_characteristic"),
        "dedup_strategy": origin.get("dedup_strategy", "mean"),
        "na_class_action": origin.get("na_class_action", "exclude"),
        "na_class_label": origin.get("na_class_label", "Unknown"),
        "use_mygene_fallback": origin.get("use_mygene_fallback", False),
        "species": origin.get("species", "human"),
        "tissue": origin["tissue"],
        "output_dir": Path(origin["output_dir"]),
    }

pd.DataFrame([
    {"gse": s["gse_id"], "tissue": s["tissue"],
     "class": s["class_characteristic"], "dedup": s["dedup_strategy"],
     "na": s["na_class_action"], "series_matrix": s["series_matrix"].name}
    for s in build_specs.values()
])

## 3. Inspect class characteristics (optional)

Before building, preview the available `!Sample_characteristics_ch1` columns and
their NA counts so you can confirm `class_characteristic` in
`configs/dataset_origin.yaml` picks a clean class column (few/zero NA, no
singleton classes). Edit `PREVIEW` to point at any buildable dataset.

In [3]:
# Pick the first buildable dataset to preview (change index to inspect another).
PREVIEW = list(build_specs.values())[0]

_, sample_meta, series_meta = parse_series_matrix(PREVIEW["series_matrix"])
print(f"{series_meta['geo_accession']} — {series_meta['title']}")
print(f"Platform: {series_meta['platform_id']}\n")

ranking = discover_class_characteristics(sample_meta)
print("Candidate class columns (ranked: 0-NA first, then larger smallest-class):")
display(ranking.drop(columns="classes"))
print("\nConfigured class_characteristic:", PREVIEW["class_characteristic"])

GSE20711 — Epigenetic portraits of human breast cancers (expression data)
Platform: GPL570

Candidate class columns (ranked: 0-NA first, then larger smallest-class):


,characteristic,n_classes,na_count,min_class_size
0,source,2,0,2
1,subtypeihc,5,0,2
2,title,90,0,1
3,methylation_barcode,90,0,1
4,size_bin,2,2,42
5,e_rfs,2,2,39
6,node,2,2,29
7,age_bin,2,2,27
8,her2_status,2,2,26
9,e_os,2,2,25



Configured class_characteristic: subtypeihc


## 4. Build all datasets

In [ ]:
# Per-origin USE_MYGENE_FALLBACK (querying MyGene.info is slower, needs internet;
# recovers only probes with a real gene — control probes like AFFX-* stay unmapped
# and are dropped by build_geo_dataset()). Override here or edit dataset_origin.yaml.

statuses = []
for spec in build_specs.values():
    if not spec["class_characteristic"]:
        print(f"SKIP {spec['gse_id']}: no class_characteristic set in dataset_origin.yaml.")
        continue
    status = build_geo_dataset(
        series_matrix_path   = spec["series_matrix"],
        class_characteristic = spec["class_characteristic"],
        annotation_path      = spec["annotation_file"],
        output_dir           = spec["output_dir"],
        tissue               = spec["tissue"],
        dedup_strategy       = spec["dedup_strategy"],
        na_class_action      = spec["na_class_action"],
        na_class_label       = spec["na_class_label"],
        use_mygene_fallback  = spec["use_mygene_fallback"],
        species              = spec["species"],
    )
    statuses.append(status)

## 5. Summary

In [ ]:
summary = pd.DataFrame([
    {
        "GEO": s["geo_id"],
        "platform": s["platform"],
        "class": s["class_characteristic"],
        "samples": s["samples_kept"],
        "probes_total": s["annotation_rows"],
        "probes_kept": s["unique_probes"],
        "genes": s["unique_genes"],
        "mapped": s["mapped_total"],
        "from_annot": s["mapped_from_annot"],
        "from_mygene": s["mapped_from_mygene"],
        "unmapped": s["unmapped"],
        "affx_removed": s["affx_removed"],
        "na_dropped": s["na_samples"],
    }
    for s in statuses
])
display(summary)

for s in statuses:
    print(f"\n{s['geo_id']} class distribution: {s['class_distribution']}")

## 6. Sanity check — outputs load through the benchmark `data_loader`

In [ ]:
for s in statuses:
    probe_csv = s["output_files"]["probe_dataset"]
    gene_csv  = s["output_files"]["gene_dataset"]

    d_probe = load_dataset(probe_csv)
    d_gene  = load_dataset(gene_csv)

    print(f"=== {s['geo_id']} ===")
    print(f"  probe : X={d_probe['X'].shape}  classes={list(d_probe['label_encoder'].classes_)}")
    print(f"  gene  : X={d_gene['X'].shape}   first genes={d_gene['feature_names'][:5]}")
    print(f"  mapping: {s['output_files']['mapping']}")
    print(f"  report : {s['output_files']['report']}\n")